In [1]:
# !pip install kaleido
# !pip install numpy pandas scikit-learn matplotlib seaborn plotly shap lime joblib scipy kaleido


In [2]:
import os
import json
import joblib
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import seaborn as sns
from datetime import datetime

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.inspection import permutation_importance

# optional libs
try:
    import shap
    SHAP_AVAILABLE = True
except Exception:
    SHAP_AVAILABLE = False

try:
    import lime
    from lime.lime_tabular import LimeTabularExplainer
    LIME_AVAILABLE = True
except Exception:
    LIME_AVAILABLE = False

try:
    import plotly.express as px
    import plotly.graph_objects as go
    PLOTLY_AVAILABLE = True
except Exception:
    PLOTLY_AVAILABLE = False


In [3]:
folder_tag = ""   # set to "_lag" to use lag artifacts
INPUT_CSV = f"artifacts/data/clean_data_with_lag_roll.csv" if folder_tag == "_lag" else "artifacts/data/clean_data.csv"
MODEL_DIR = f"artifacts/models{folder_tag}/"
PREPROCESSOR_PATH = f"artifacts/preprocessor/preprocessor.pkl"   # same path used earlier
FEATURE_SCHEMA_PATH = f"feature_schema{folder_tag}.json"
RESULT_DIR = f"artifacts/analysis{folder_tag}/"
os.makedirs(RESULT_DIR, exist_ok=True)
os.makedirs(os.path.join(RESULT_DIR, "plots"), exist_ok=True)
os.makedirs(os.path.join(RESULT_DIR, "shap"), exist_ok=True)
os.makedirs(os.path.join(RESULT_DIR, "perm_importance"), exist_ok=True)
os.makedirs(os.path.join(RESULT_DIR, "ablation"), exist_ok=True)
os.makedirs(os.path.join(RESULT_DIR, "topk"), exist_ok=True)

In [4]:
TARGET = "Average_Price"
TOPK = 5

In [5]:
# -------------------------
# === Helpers ===
# -------------------------
def regression_metrics(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    # safe MAPE
    mask = y_true != 0
    mape = np.nan
    if mask.sum() > 0:
        mape = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100
    r2 = r2_score(y_true, y_pred)
    return {"RMSE": rmse, "MSE": mse, "MAE": mae, "MAPE": mape, "R2": r2}


In [6]:
def load_models(model_dir=MODEL_DIR):
    files = os.listdir(model_dir) if os.path.exists(model_dir) else []
    models = {}
    for fn in files:
        if fn.endswith("_best.joblib") or fn.endswith(".joblib"):
            path = os.path.join(model_dir, fn)
            name = fn.replace("_best.joblib", "").replace(".joblib", "")
            try:
                models[name] = joblib.load(path)
            except Exception as e:
                print(f"Failed to load {path}: {e}")
    return models

In [7]:
def load_preprocessor(path=PREPROCESSOR_PATH):
    if os.path.exists(path):
        return joblib.load(path)
    else:
        print("Preprocessor not found at", path)
        return None

def safe_predict(model, X):
    """Return predictions for sklearn-style pipelines and simple estimators."""
    if hasattr(model, "predict"):
        return model.predict(X)
    else:
        raise RuntimeError("Model does not have predict")

In [8]:
# -------------------------
# === Load assets ===
# -------------------------
print("Loading data and artifacts...")
df = pd.read_csv(INPUT_CSV, parse_dates=["Date"], infer_datetime_format=True)
df = df.sort_values("Date").reset_index(drop=True)
preprocessor = load_preprocessor()  # may be None
with open(FEATURE_SCHEMA_PATH, "r") as f:
    feature_schema = json.load(f)

models = load_models()
print("Found models:", list(models.keys()))
print("SHAP available:", SHAP_AVAILABLE, "LIME available:", LIME_AVAILABLE, "Plotly:", PLOTLY_AVAILABLE)


Loading data and artifacts...
Found models: ['arima_baseline', 'LightGBM', 'Linear', 'RandomForest', 'XGBoost']
SHAP available: True LIME available: True Plotly: True


In [9]:
# create features & target
FEATURES = feature_schema
if "Date" in FEATURES:
    FEATURES = [c for c in FEATURES if c != "Date"]
X_all = df[FEATURES].copy()
y_all = df[TARGET].copy()

# if preprocessor exists, transform
if preprocessor is not None:
    X_preprocessed = preprocessor.transform(X_all)
else:
    X_preprocessed = X_all.values

In [10]:
# We'll produce a holdout: last 20% as earlier
holdout_frac = 0.2
n_hold = max(1, int(len(df) * holdout_frac))
train_idx = slice(0, len(df) - n_hold)
test_idx = slice(len(df) - n_hold, len(df))
X_train = X_all.iloc[train_idx].reset_index(drop=True)
y_train = y_all.iloc[train_idx].reset_index(drop=True)
X_test = X_all.iloc[test_idx].reset_index(drop=True)
y_test = y_all.iloc[test_idx].reset_index(drop=True)

if preprocessor is not None:
    X_train_pp = preprocessor.transform(X_train)
    X_test_pp = preprocessor.transform(X_test)
else:
    X_train_pp = X_train.values
    X_test_pp = X_test.values

In [11]:

# -------------------------
# === 1) Evaluate models on holdout ===
# -------------------------
print("\nEvaluating models on holdout...")
eval_records = []
predictions = {}
for name, model in models.items():
    try:
        # Fit-check: if model is an sklearn pipeline already fitted, skip; else fit on train
        needs_fit = False
        try:
            # some joblib-saved pipelines are already fit
            _ = getattr(model, "predict")
            if hasattr(model, "steps"):  # sklearn pipeline
                # pipeline might be fitted already; we try predict, if fails we fit
                try:
                    yhat = safe_predict(model, X_test)
                except Exception:
                    needs_fit = True
            else:
                # non-pipeline estimator
                try:
                    _ = model.predict(X_test_pp)
                except Exception:
                    needs_fit = True
        except Exception:
            needs_fit = True
        if needs_fit:
            print(f"Fitting model {name} on full training portion...")
            if hasattr(model, "fit"):
                # prefer to fit using original pipeline and preprocessor
                try:
                    model.fit(X_train, y_train)
                except Exception:
                    # try fitted on preprocessed arrays
                    model.fit(X_train_pp, y_train)
            else:
                print("Cannot fit model:", name)

        # Predict (if pipeline expects raw df, pass X_test; else pass preprocessed)
        try:
            # If it's a pipeline with preproc that expects raw features, predict with raw X_test
            yhat = safe_predict(model, X_test)
        except Exception:
            # fallback to preprocessed arrays
            yhat = safe_predict(model, X_test_pp)

        predictions[name] = np.array(yhat)
        mets = regression_metrics(y_test, yhat)
        mets["Model"] = name
        eval_records.append(mets)
        print(f"{name} → RMSE: {mets['RMSE']:.4f}  MAE: {mets['MAE']:.4f}  R2: {mets['R2']:.4f}")
    except Exception as e:
        print("Failed eval for", name, e)

eval_df = pd.DataFrame(eval_records).sort_values("RMSE")
eval_df.to_csv(os.path.join(RESULT_DIR, "holdout_performance.csv"), index=False)
print("Saved holdout_performance.csv")


Evaluating models on holdout...
Fitting model arima_baseline on full training portion...
Cannot fit model: arima_baseline
Failed eval for arima_baseline Cannot convert input [[[-0.50593194  1.20663806 -0.23638547 ... -0.63373065  0.68083528
   1.31095075]
 [-0.50593194  1.22411791 -0.23638547 ...  1.57795746  0.68083528
   1.31095075]
 [-0.50593194  1.22411791 -0.23638547 ...  1.57795746  0.68083528
   1.31095075]
 ...
 [-0.62020713  1.64946087 -0.92400729 ...  1.57795746 -1.22742422
   0.78546697]
 [-0.62020713  1.64946087 -0.92400729 ...  1.57795746 -1.22742422
   0.78546697]
 [-0.62020713  1.64946087 -0.92400729 ... -0.63373065 -1.22742422
   0.78546697]]] of type <class 'numpy.ndarray'> to Timestamp


LightGBM → RMSE: 9.3748  MAE: 7.2116  R2: 0.6811
Linear → RMSE: 13.2554  MAE: 10.8407  R2: 0.3624
RandomForest → RMSE: 7.6826  MAE: 5.2825  R2: 0.7858
XGBoost → RMSE: 11.7962  MAE: 9.0345  R2: 0.4951
Saved holdout_performance.csv


In [12]:
def plot_prediction_kde(y_true, predictions, result_dir, target_name="AveragePrice"):
    """
    Creates KDE comparison plots between actual values and each model's predictions.
    - y_true: array-like of actual target values
    - predictions: dict {model_name: y_pred array}
    - result_dir: output directory
    """

    os.makedirs(os.path.join(result_dir, "plots"), exist_ok=True)

    # --- Matplotlib version ---
    plt.figure(figsize=(10, 6))
    sns.kdeplot(y_true, fill=False, color="black", lw=3, label="Actual")
    for name, yhat in predictions.items():
        sns.kdeplot(yhat, fill=True, alpha=0.15, lw=1.2, label=name)
    plt.title(f"KDE Distribution — {target_name} vs Model Predictions", fontsize=14)
    plt.xlabel(target_name)
    plt.ylabel("Density")
    plt.legend()
    plt.tight_layout()

    kde_img_path = os.path.join(result_dir, "plots", f"kde_comparison_{target_name}.png")
    plt.savefig(kde_img_path, dpi=300)
    plt.close()
    print(f"Saved KDE Matplotlib plot: {kde_img_path}")

    # --- Plotly version ---
    fig = go.Figure()

    # Actual curve (no fill, strong line)
    fig.add_trace(go.Violin(y=y_true, name="Actual", box_visible=False, meanline_visible=True,
                            line_color="black", fillcolor="rgba(0,0,0,0)", opacity=1, width=0.8))

    # Add KDE-like scatter traces per model (approximated smoothness)
    for name, yhat in predictions.items():
        kde = sns.kdeplot(yhat)
        x, y = kde.get_lines()[-1].get_data()
        fig.add_trace(go.Scatter(x=x, y=y, fill='tozeroy', name=name,
                                 mode='lines', line=dict(width=1.5), opacity=0.4))
        plt.close()

    fig.update_layout(
        title=f"KDE Distribution Comparison — {target_name}",
        xaxis_title=target_name,
        yaxis_title="Density",
        hovermode="x unified"
    )

    kde_html_path = os.path.join(result_dir, "plots", f"kde_comparison_{target_name}.html")
    fig.write_html(kde_html_path)
    print(f"Saved interactive KDE Plotly plot: {kde_html_path}")


In [14]:
import matplotlib.pyplot as plt
plot_prediction_kde(y_test, predictions, RESULT_DIR, target_name=TARGET)

Saved KDE Matplotlib plot: artifacts/analysis/plots\kde_comparison_Average_Price.png
Saved interactive KDE Plotly plot: artifacts/analysis/plots\kde_comparison_Average_Price.html


In [15]:
# -------------------------
# === 2) Permutation importance (global) ===
# -------------------------
print("\nRunning permutation importance (sklearn) per model...")
perm_summary = []
for name, model in models.items():
    try:
        # choose inputs for permutation: preprocessed array & feature names
        # Many pipelines wrap preprocessor; we want to feed the correct X to permutation_importance.
        try:
            # if model is a pipeline and includes preproc, then permutation_importance expects X raw
            if hasattr(model, "predict") and hasattr(model, "steps") and preprocessor is not None:
                X_for_perm = X_test.copy()
                r = permutation_importance(model, X_for_perm, y_test, n_repeats=10, random_state=42, n_jobs=1)
                feat_names = FEATURES
            else:
                # use preprocessed arrays and feature names after preprocessing (if preprocessor exists)
                X_for_perm = X_test_pp
                r = permutation_importance(model, X_for_perm, y_test, n_repeats=10, random_state=42, n_jobs=1)
                # feature names fallback to FEATURES (note: one-hot expands names but here we use original)
                feat_names = FEATURES
        except Exception:
            # fallback: try preprocessed
            X_for_perm = X_test_pp
            r = permutation_importance(model, X_for_perm, y_test, n_repeats=10, random_state=42, n_jobs=1)
            feat_names = FEATURES

        importances = pd.DataFrame({
            "feature": feat_names,
            "importance_mean": r.importances_mean,
            "importance_std": r.importances_std
        }).sort_values("importance_mean", ascending=False)

        importances.to_csv(os.path.join(RESULT_DIR, "perm_importance", f"{name}_perm_importance.csv"), index=False)
        perm_summary.append({"Model": name, "Top1": importances.iloc[0]["feature"] if not importances.empty else None})
        print(f"Saved permutation importance for {name}")
    except Exception as e:
        print("Permutation importance failed for", name, e)

pd.DataFrame(perm_summary).to_csv(os.path.join(RESULT_DIR, "perm_importance", "summary.csv"), index=False)



Running permutation importance (sklearn) per model...
Permutation importance failed for arima_baseline The 'estimator' parameter of permutation_importance must be an object implementing 'fit'. Got <statsmodels.tsa.arima.model.ARIMAResultsWrapper object at 0x000002DDD45F6A50> instead.
Saved permutation importance for LightGBM
Saved permutation importance for Linear
Saved permutation importance for RandomForest
Saved permutation importance for XGBoost


In [16]:
import os
import matplotlib.pyplot as plt
import seaborn as sns

# --- ensure directory exists ---
perm_dir = os.path.join(RESULT_DIR, "perm_importance")
os.makedirs(perm_dir, exist_ok=True)

# --- save CSV summary ---
pd.DataFrame(perm_summary).to_csv(os.path.join(perm_dir, "summary.csv"), index=False)

# --- plot top 10 features per model ---
for name, model in models.items():
    try:
        df_imp = pd.read_csv(os.path.join(perm_dir, f"{name}_perm_importance.csv"))
        top_n = df_imp.head(10)
        
        fig, ax = plt.subplots(figsize=(8,6))
        sns.barplot(x="importance_mean", y="feature", data=top_n, ax=ax, palette="viridis")
        ax.set_title(f"Permutation Importance — Top 10 Features ({name})")
        ax.set_xlabel("Mean Importance")
        ax.set_ylabel("Feature")
        plt.tight_layout()
        plot_file = os.path.join(perm_dir, f"{name}_perm_importance.png")
        plt.savefig(plot_file, dpi=200)
        plt.close(fig)
        print("Saved permutation importance plot:", plot_file)
    except Exception as e:
        print("Failed to plot permutation importance for", name, e)


Failed to plot permutation importance for arima_baseline [Errno 2] No such file or directory: 'artifacts/analysis/perm_importance\\arima_baseline_perm_importance.csv'
Saved permutation importance plot: artifacts/analysis/perm_importance\LightGBM_perm_importance.png
Saved permutation importance plot: artifacts/analysis/perm_importance\Linear_perm_importance.png
Saved permutation importance plot: artifacts/analysis/perm_importance\RandomForest_perm_importance.png
Saved permutation importance plot: artifacts/analysis/perm_importance\XGBoost_perm_importance.png


In [17]:
os.makedirs(os.path.join(RESULT_DIR, "ablation"), exist_ok=True)

# Load your feature groups
import pickle
with open(f"col_categories{folder_tag}.joblib", "rb") as f:
    feature_groups = pickle.load(f)

# Helper function to compute metrics
def regression_metrics(y_true, y_pred):
    from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    return {"RMSE": rmse, "MAE": mae, "R2": r2}

# List of sklearn-compatible models only
sklearn_models = {name: model for name, model in models.items() if "arima" not in name.lower()}

# ===== 1) Category-based ablation =====
cat_records = []

for group_name, cols in feature_groups.items():
    print(f"\nCategory Ablation: dropping {group_name} ({len(cols)} features)")
    # Only drop columns present in X_train
    cols_to_drop = [c for c in cols if c in X_train.columns]
    if not cols_to_drop:
        print(f"No columns found for group {group_name}, skipping.")
        continue

    remaining = [c for c in X_train.columns if c not in cols_to_drop]

    for name, model in sklearn_models.items():
        try:
            X_train_sub = X_train[remaining].reset_index(drop=True)
            X_test_sub = X_test[remaining].reset_index(drop=True)

            est_clone = clone(model)
            est_clone.fit(X_train_sub, y_train)
            yhat = est_clone.predict(X_test_sub)

            mets = regression_metrics(y_test, yhat)
            mets.update({"Model": name, "Dropped_Group": group_name})
            cat_records.append(mets)

        except Exception as e:
            print(f"Category ablation failed for {name}, group {group_name}: {e}")

cat_df = pd.DataFrame(cat_records)
cat_df.to_csv(os.path.join(RESULT_DIR, "ablation", "category_ablation.csv"), index=False)
print("Saved category_ablation.csv")

# Heatmap for category ablation (RMSE)
plt.figure(figsize=(10,6))
if not cat_df.empty:
    cat_pivot = cat_df.pivot(index="Model", columns="Dropped_Group", values="RMSE")
    sns.heatmap(cat_pivot, annot=True, fmt=".2f", cmap="YlGnBu")
    plt.title("Category-based Ablation: RMSE per Model")
    plt.tight_layout()
    plt.savefig(os.path.join(RESULT_DIR, "ablation", "category_ablation_heatmap.png"))
    plt.show()

# ===== 2) Individual feature ablation =====
ind_records = []

for feat in X_train.columns:
    print(f"\nIndividual feature ablation: dropping {feat}")
    remaining = [c for c in X_train.columns if c != feat]

    for name, model in sklearn_models.items():
        try:
            X_train_sub = X_train[remaining].reset_index(drop=True)
            X_test_sub = X_test[remaining].reset_index(drop=True)

            est_clone = clone(model)
            est_clone.fit(X_train_sub, y_train)
            yhat = est_clone.predict(X_test_sub)

            mets = regression_metrics(y_test, yhat)
            mets.update({"Model": name, "Dropped_Feature": feat})
            ind_records.append(mets)

        except Exception as e:
            print(f"Feature ablation failed for {name}, feature {feat}: {e}")

ind_df = pd.DataFrame(ind_records)
ind_df.to_csv(os.path.join(RESULT_DIR, "ablation", "individual_feature_ablation.csv"), index=False)
print("Saved individual_feature_ablation.csv")

# Bar plot for individual features per model
for name in ind_df["Model"].unique():
    plt.figure(figsize=(12,6))
    tmp = ind_df[ind_df["Model"]==name].sort_values("RMSE", ascending=False)
    sns.barplot(x="RMSE", y="Dropped_Feature", data=tmp, palette="viridis")
    plt.title(f"Individual Feature Ablation - {name} (higher RMSE = more important)")
    plt.xlabel("RMSE after dropping feature")
    plt.tight_layout()
    plt.savefig(os.path.join(RESULT_DIR, "ablation", f"{name}_individual_ablation.png"))
    plt.show()


Category Ablation: dropping climate_cols (20 features)
Category ablation failed for LightGBM, group climate_cols: name 'clone' is not defined
Category ablation failed for Linear, group climate_cols: name 'clone' is not defined
Category ablation failed for RandomForest, group climate_cols: name 'clone' is not defined
Category ablation failed for XGBoost, group climate_cols: name 'clone' is not defined

Category Ablation: dropping economic_cols (3 features)
Category ablation failed for LightGBM, group economic_cols: name 'clone' is not defined
Category ablation failed for Linear, group economic_cols: name 'clone' is not defined
Category ablation failed for RandomForest, group economic_cols: name 'clone' is not defined
Category ablation failed for XGBoost, group economic_cols: name 'clone' is not defined

Category Ablation: dropping market_cols (1 features)
Category ablation failed for LightGBM, group market_cols: name 'clone' is not defined
Category ablation failed for Linear, group mar

KeyError: 'Model'

<Figure size 1000x600 with 0 Axes>

In [18]:
# Robust ablation that accounts for ColumnTransformer internals and 'num__' style features.
import os, copy
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.exceptions import NotFittedError

sns.set(style="whitegrid")
os.makedirs(os.path.join(RESULT_DIR, "ablation"), exist_ok=True)

# load feature groups (your dict already saved)
import joblib
feature_groups = joblib.load(f"col_categories{folder_tag}.joblib")

def regression_metrics(y_true, y_pred):
    from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    return {"RMSE": rmse, "MAE": mae, "R2": r2}

# helper: build a filtered ColumnTransformer from an existing one using remaining columns
from sklearn.base import BaseEstimator, TransformerMixin

def build_filtered_preprocessor(orig_preproc: ColumnTransformer, remaining_cols):
    """
    Returns a new ColumnTransformer with the same transformer objects (cloned)
    but with column lists filtered to only include remaining_cols.
    """
    new_transformers = []
    for name, transformer, cols in orig_preproc.transformers:
        # sklearn ColumnTransformer may include ('remainder', 'drop' or transformer, cols)
        if name == 'remainder':
            # preserve remainder setting
            new_transformers.append((name, transformer, cols))
            continue

        cols_filtered = [c for c in cols if c in remaining_cols]
        # if no columns remain for this transformer, skip it
        if len(cols_filtered) == 0:
            continue

        # clone the transformer pipeline or transformer
        try:
            trans_clone = clone(transformer)
        except Exception:
            # fallback: use original transformer (should still work but be careful)
            trans_clone = transformer

        new_transformers.append((name, trans_clone, cols_filtered))

    # If no transformers remain, return None
    if len(new_transformers) == 0:
        return None

    new_pre = ColumnTransformer(new_transformers, remainder="drop", sparse_threshold=0)
    return new_pre

# select sklearn-compatible models (skip statsmodels / ARIMA)
sklearn_models = {}
for name, model in models.items():
    # skip objects that are not sklearn-like pipelines/estimators
    try:
        _ = clone(model)
        sklearn_models[name] = model
    except Exception:
        # sometimes pipeline clone fails but model is pipeline; we still want pipelines
        if hasattr(model, "named_steps"):
            sklearn_models[name] = model
        else:
            print("Skipping non-sklearn model entirely:", name)

print("Models considered for ablation:", list(sklearn_models.keys()))

# Function to get pipeline's original input columns (if pipeline exists)
def get_preproc_input_columns(model):
    if hasattr(model, "named_steps") and "preproc" in model.named_steps:
        pre = model.named_steps["preproc"]
        # transformers may be stored in pre.transformers (list of tuples)
        cols = []
        for name, transformer, c in getattr(pre, "transformers", []):
            # skip 'remainder'
            if name == "remainder":
                continue
            # some designers use empty list for categorical pipeline; that's ok
            cols.extend([col for col in c])
        return cols
    else:
        # no pipeline preprocessor - assume model expects raw X_train columns
        return list(X_train.columns)

# =========================
# Category-based ablation
# =========================
cat_records = []
for group_name, cols in feature_groups.items():
    print(f"\nCategory ablation: group = {group_name} (declared len {len(cols)})")
    # ensure cols is a list
    cols_to_drop_declared = [c for c in cols]
    for name, model in sklearn_models.items():
        try:
            # get the actual input columns this model's preprocessor expects
            input_cols = get_preproc_input_columns(model)
            # intersect with declared drop list
            cols_to_drop = [c for c in cols_to_drop_declared if c in input_cols]
            if not cols_to_drop:
                # nothing to drop for this model (maybe group columns not used in this pipeline)
                print(f"  {name}: nothing to drop (no overlap with model input cols)")
                continue

            remaining_cols = [c for c in input_cols if c not in cols_to_drop]
            if len(remaining_cols) == 0:
                print(f"  {name}: no remaining cols after dropping -> skip")
                continue

            # If model is pipeline with preproc, build filtered preprocessor
            if hasattr(model, "named_steps") and "preproc" in model.named_steps:
                orig_pre = model.named_steps["preproc"]
                est_final = model.named_steps.get("est", None)
                # build filtered pre
                new_pre = build_filtered_preprocessor(orig_pre, remaining_cols)
                if new_pre is None:
                    print(f"  {name}: filtered preprocessor empty -> skipping")
                    continue
                # assemble new pipeline: (filtered_pre, cloned estimator)
                try:
                    est_clone = clone(est_final) if est_final is not None else None
                except Exception:
                    est_clone = est_final
                if est_clone is None:
                    print(f"  {name}: cannot clone final estimator - skipping")
                    continue
                new_pipe = Pipeline([("preproc", new_pre), ("est", est_clone)])
                # Fit on raw X_train but only columns that the original preprocessor used (remaining_cols)
                Xtr_sub = X_train[remaining_cols].reset_index(drop=True)
                Xte_sub = X_test[remaining_cols].reset_index(drop=True)
                new_pipe.fit(Xtr_sub, y_train)
                yhat = new_pipe.predict(Xte_sub)
            else:
                # model is plain estimator that expects raw DataFrame -> just drop columns from X_train
                Xtr_sub = X_train.drop(columns=cols_to_drop, errors="ignore").reset_index(drop=True)
                Xte_sub = X_test.drop(columns=cols_to_drop, errors="ignore").reset_index(drop=True)
                est_clone = clone(model)
                est_clone.fit(Xtr_sub, y_train)
                yhat = est_clone.predict(Xte_sub)

            mets = regression_metrics(y_test, yhat)
            mets.update({"Model": name, "Dropped_Group": group_name, "Dropped_Count": len(cols_to_drop)})
            cat_records.append(mets)
            print(f"  {name}: done (dropped {len(cols_to_drop)} cols)")

        except Exception as e:
            print(f"  {name}: category ablation failed for group {group_name}: {repr(e)}")

cat_df = pd.DataFrame(cat_records)
cat_df.to_csv(os.path.join(RESULT_DIR, "ablation", "category_ablation.csv"), index=False)
print("\nSaved category_ablation.csv ->", os.path.join(RESULT_DIR, "ablation", "category_ablation.csv"))

# Heatmap (if results exist)
if not cat_df.empty:
    pivot = cat_df.pivot(index="Model", columns="Dropped_Group", values="RMSE")
    plt.figure(figsize=(max(6, pivot.shape[1]*1.2), max(4, pivot.shape[0]*0.8)))
    sns.heatmap(pivot, annot=True, fmt=".3f", cmap="coolwarm", center=None)
    plt.title("Category Ablation: RMSE when dropping each category")
    plt.tight_layout()
    png = os.path.join(RESULT_DIR, "ablation", "category_ablation_heatmap.png")
    plt.savefig(png, dpi=200)
    plt.close()
    print("Saved heatmap:", png)
else:
    print("No category ablation results to plot.")

# =========================
# Individual feature ablation
# =========================
ind_records = []
# We'll iterate features from the union of all models' input columns to ensure coverage
all_input_cols = set()
for name, model in sklearn_models.items():
    all_input_cols.update(get_preproc_input_columns(model))
all_input_cols = [c for c in all_input_cols if c in X_train.columns]

print("\nIndividual ablation across features (count):", len(all_input_cols))

for feat in all_input_cols:
    print("Dropping feature:", feat)
    for name, model in sklearn_models.items():
        try:
            input_cols = get_preproc_input_columns(model)
            if feat not in input_cols:
                # this model didn't use this feature originally
                continue
            remaining_cols = [c for c in input_cols if c != feat]
            if len(remaining_cols) == 0:
                continue

            if hasattr(model, "named_steps") and "preproc" in model.named_steps:
                orig_pre = model.named_steps["preproc"]
                est_final = model.named_steps.get("est", None)

                new_pre = build_filtered_preprocessor(orig_pre, remaining_cols)
                if new_pre is None:
                    continue
                try:
                    est_clone = clone(est_final) if est_final is not None else None
                except Exception:
                    est_clone = est_final
                if est_clone is None:
                    continue

                new_pipe = Pipeline([("preproc", new_pre), ("est", est_clone)])
                Xtr_sub = X_train[remaining_cols].reset_index(drop=True)
                Xte_sub = X_test[remaining_cols].reset_index(drop=True)
                new_pipe.fit(Xtr_sub, y_train)
                yhat = new_pipe.predict(Xte_sub)
            else:
                Xtr_sub = X_train.drop(columns=[feat], errors="ignore").reset_index(drop=True)
                Xte_sub = X_test.drop(columns=[feat], errors="ignore").reset_index(drop=True)
                est_clone = clone(model)
                est_clone.fit(Xtr_sub, y_train)
                yhat = est_clone.predict(Xte_sub)

            mets = regression_metrics(y_test, yhat)
            mets.update({"Model": name, "Dropped_Feature": feat})
            ind_records.append(mets)
        except Exception as e:
            print(f"  {name}: individual ablation failed for feature {feat}: {repr(e)}")

ind_df = pd.DataFrame(ind_records)
ind_df.to_csv(os.path.join(RESULT_DIR, "ablation", "individual_feature_ablation.csv"), index=False)
print("Saved individual_feature_ablation.csv ->", os.path.join(RESULT_DIR, "ablation", "individual_feature_ablation.csv"))

# Per-model barplots of top-k worst features (by RMSE increase)
if not ind_df.empty:
    for name in ind_df["Model"].unique():
        tmp = ind_df[ind_df["Model"]==name].copy()
        # compute delta RMSE vs baseline (baseline metric from eval_df if available)
        try:
            baseline_rmse = float(eval_df[eval_df["Model"]==name]["RMSE"].iloc[0])
        except Exception:
            baseline_rmse = tmp["RMSE"].min()  # fallback
        tmp["delta_RMSE"] = tmp["RMSE"] - baseline_rmse
        # sort by delta_RMSE descending
        top = tmp.sort_values("delta_RMSE", ascending=False).head(30)
        plt.figure(figsize=(10, max(4, len(top)*0.25)))
        sns.barplot(x="delta_RMSE", y="Dropped_Feature", data=top, palette="viridis")
        plt.axvline(0, color="k", linewidth=0.6)
        plt.title(f"{name}: Individual feature ablation (ΔRMSE vs baseline)")
        plt.xlabel("ΔRMSE (higher = more important)")
        plt.tight_layout()
        outp = os.path.join(RESULT_DIR, "ablation", f"{name}_individual_ablation_delta.png")
        plt.savefig(outp, dpi=200)
        plt.close()
        print("Saved:", outp)
else:
    print("No individual ablation results to plot.")


Skipping non-sklearn model entirely: arima_baseline
Models considered for ablation: ['LightGBM', 'Linear', 'RandomForest', 'XGBoost']

Category ablation: group = climate_cols (declared len 20)
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000383 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 660
[LightGBM] [Info] Number of data points in the train set: 1112, number of used features: 15
[LightGBM] [Info] Start training from score 66.550395
  LightGBM: done (dropped 20 cols)
  Linear: done (dropped 20 cols)
  RandomForest: done (dropped 20 cols)
  XGBoost: done (dropped 20 cols)

Category ablation: group = economic_cols (declared len 3)
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000746 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set

In [19]:
# -------------------------
# === 4) SHAP (global & local, Top-5 Waterfalls) ===
# -------------------------
if SHAP_AVAILABLE:
    print("\nRunning SHAP analyses...")
    os.makedirs(os.path.join(RESULT_DIR, "shap"), exist_ok=True)

    for name, model in models.items():
        try:
            print("\nSHAP for", name)
            est = model
            X_for_shap = None
            shap_feature_names = None

            # prepare data for SHAP
            if hasattr(model, "steps") and preprocessor is not None:
                try:
                    est = model.named_steps["est"]
                    X_for_shap = preprocessor.transform(X_test)
                    shap_feature_names = feature_schema
                except Exception:
                    X_for_shap = X_test.copy()
                    shap_feature_names = X_test.columns.tolist()
            else:
                if preprocessor is not None:
                    X_for_shap = X_test_pp
                    shap_feature_names = feature_schema
                else:
                    X_for_shap = X_test.copy()
                    shap_feature_names = X_test.columns.tolist()

            # pick SHAP explainer
            try:
                if any(t in name for t in ["RandomForest", "XGBoost", "LightGBM"]) and hasattr(est, "predict"):
                    explainer = shap.TreeExplainer(est)
                elif "Linear" in name:
                    explainer = shap.LinearExplainer(est, X_for_shap, feature_perturbation="interventional")
                else:
                    explainer = shap.KernelExplainer(lambda x: est.predict(x), shap.sample(X_for_shap, min(100, len(X_for_shap))))
            except Exception:
                explainer = shap.KernelExplainer(lambda x: model.predict(x), shap.sample(X_for_shap, min(100, len(X_for_shap))))

            shap_values = explainer.shap_values(X_for_shap)

            # --- GLOBAL SUMMARY ---
            try:
                out_png = os.path.join(RESULT_DIR, "shap", f"{name}_summary.png")
                shap.summary_plot(shap_values, features=X_for_shap, feature_names=shap_feature_names, show=False)
                import matplotlib.pyplot as plt
                plt.savefig(out_png, bbox_inches="tight", dpi=200)
                plt.close()
                print("Saved SHAP summary:", out_png)
            except Exception as e:
                print("Failed to save shap summary plot:", e)

            # --- Save SHAP values ---
            try:
                sv = np.array(shap_values)
                if sv.ndim == 2:
                    sv_df = pd.DataFrame(sv, columns=shap_feature_names)
                else:
                    sv_df = pd.DataFrame(np.abs(sv).sum(axis=0), index=shap_feature_names).T
                sv_df.to_csv(os.path.join(RESULT_DIR, "shap", f"{name}_shap_values.csv"), index=False)
            except Exception as e:
                print("Failed to write shap values:", e)

            # --- LOCAL EXPLANATIONS: TOP-5 WORST CASES ---
            try:
                try:
                    yhat_test = model.predict(X_test)
                except Exception:
                    yhat_test = model.predict(X_test_pp)

                errors = np.abs(y_test.values - yhat_test)
                top5_idx = np.argsort(errors)[-5:][::-1]  # top 5 largest errors

                for rank, idx in enumerate(top5_idx, 1):
                    try:
                        shap_exp = shap.Explanation(
                            values=shap_values[idx],
                            base_values=explainer.expected_value,
                            data=X_for_shap[idx],
                            feature_names=shap_feature_names,
                        )
                        shap.plots.waterfall(shap_exp, show=False)
                        out_wf = os.path.join(RESULT_DIR, "shap", f"{name}_waterfall_{rank}.png")
                        plt.savefig(out_wf, bbox_inches="tight", dpi=200)
                        plt.close()
                        print(f"Saved waterfall #{rank}:", out_wf)
                    except Exception as e:
                        print(f"  Failed to save waterfall #{rank} for {name}: {e}")
            except Exception as e:
                print("Failed local SHAP waterfall generation for", name, e)

        except Exception as e:
            print("SHAP failed for", name, e)

else:
    print("SHAP not installed — skipping SHAP analyses. Install shap via `pip install shap` to enable.")

# short SHAP vs waterfall note
with open(os.path.join(RESULT_DIR, "shap_explainers_note.txt"), "w") as f:
    f.write("SHAP summary (beeswarm) -> global feature influence across dataset.\n")
    f.write("Waterfall -> local breakdown for a single sample.\n")
    f.write("Now generated top-5 waterfalls per model (largest absolute prediction errors).\n")



Running SHAP analyses...

SHAP for arima_baseline
Provided model function fails when applied to the provided data set.
Provided model function fails when applied to the provided data set.
SHAP failed for arima_baseline Cannot convert input [[[-0.64578756  1.51156429 -0.61839759 ...  1.57795746 -1.22742422
  -0.65018142]
 [-0.37683443  1.71743806 -0.61839759 ...  1.57795746 -1.22742422
  -0.65018142]
 [-0.4342111   1.19498483 -0.69480001 ...  1.57795746 -0.71610763
  -1.17566521]
 ...
 [-0.52386215  0.87063654 -0.61839759 ... -0.63373065  1.19215187
  -0.65018142]
 [-0.52983888  0.92113387 -0.92400729 ... -0.63373065  0.68083528
  -1.17566521]
 [-0.53270772  1.90388977 -0.92400729 ... -0.63373065 -1.22742422
   0.78546697]]] of type <class 'numpy.ndarray'> to Timestamp

SHAP for LightGBM
Saved SHAP summary: artifacts/analysis/shap\LightGBM_summary.png
Saved waterfall #1: artifacts/analysis/shap\LightGBM_waterfall_1.png
Saved waterfall #2: artifacts/analysis/shap\LightGBM_waterfall_2.pn

In [20]:
# -------------------------
# === 5) Top-K error analysis + local explanations (SHAP or LIME) ===
# -------------------------
print("\nTop-K error analysis (TopK = {})".format(TOPK))
topk_records = []
for name, yhat in predictions.items():
    try:
        errors = np.abs(y_test.values - yhat)
        worst_idx = np.argsort(-errors)[:TOPK]
        recs = []
        for rank, idx in enumerate(worst_idx, 1):
            rec = {
                "Model": name,
                "Rank": rank,
                "Index_in_holdout": int(idx),
                "Date": str(df["Date"].iloc[test_idx].reset_index(drop=True).iloc[idx]),
                "Actual": float(y_test.iloc[idx]),
                "Predicted": float(yhat[idx]),
                "AbsError": float(errors[idx]),
                "SignedError": float(yhat[idx] - y_test.iloc[idx])
            }
            recs.append(rec)
        pd.DataFrame(recs).to_csv(os.path.join(RESULT_DIR, "topk", f"{name}_topk_errors.csv"), index=False)
        topk_records.extend(recs)
        print(f"Saved top-k errors for {name}")
        # Local explanations for each top-k record
        if SHAP_AVAILABLE:
            # attempt SHAP local explanation for each point
            try:
                # reuse previous shap workflow where possible; produce explainer if not exist
                # For speed, we won't re-create explainer here — user should run full shap block above if they want details
                pass
            except Exception:
                pass
        elif LIME_AVAILABLE:
            # run LIME for the top-k points
            try:
                # create LIME explainer using training data (preprocessed or raw)
                if preprocessor is not None:
                    train_arr = X_train_pp
                    feature_names = FEATURES
                else:
                    train_arr = X_train.values
                    feature_names = X_train.columns.tolist()
                explainer = LimeTabularExplainer(train_arr, feature_names=feature_names, verbose=False, mode="regression")
                for r in recs:
                    idx = r["Index_in_holdout"]
                    if preprocessor is not None:
                        instance = X_test_pp[idx]
                        predict_fn = lambda x: models[name].predict(x) if hasattr(models[name], "predict") else models[name].predict(x)
                    else:
                        instance = X_test.iloc[idx].values
                        predict_fn = lambda x: models[name].predict(x)
                    exp = explainer.explain_instance(instance, predict_fn, num_features=10)
                    out_html = os.path.join(RESULT_DIR, "topk", f"{name}_lime_top{r['Rank']}_idx{idx}.html")
                    exp.save_to_file(out_html)
            except Exception as e:
                print("LIME failed:", e)
    except Exception as e:
        print("TopK analysis failed for", name, e)

pd.DataFrame(topk_records).to_csv(os.path.join(RESULT_DIR, "topk", "topk_summary.csv"), index=False)



Top-K error analysis (TopK = 5)
Saved top-k errors for LightGBM
Saved top-k errors for Linear
Saved top-k errors for RandomForest
Saved top-k errors for XGBoost


In [25]:
# -------------------------
# === 6) Prediction distributions & overshoot/undershoot summary ===
# -------------------------
dist_records = []
for name, yhat in predictions.items():
    try:
        lows = np.percentile(yhat, 5)
        highs = np.percentile(yhat, 95)
        median = np.median(yhat)
        overshoot = np.sum(yhat > y_test.values)
        undershoot = np.sum(yhat < y_test.values)
        dist_records.append({
            "Model": name, "5p": lows, "95p": highs, "median": median,
            "overshoot_count": int(overshoot), "undershoot_count": int(undershoot),
            "overshoot_prop": float(overshoot / len(yhat)), "undershoot_prop": float(undershoot / len(yhat))
        })
        # interactive histogram
        if PLOTLY_AVAILABLE:
            fig = px.histogram(x=yhat, nbins=50, title=f"{name} Prediction Distribution (holdout)")
            fig.update_layout(xaxis_title=TARGET, yaxis_title="count")
            pfile = os.path.join(RESULT_DIR, "plots", f"{name}_pred_distribution.html")
            fig.write_html(pfile)
    except Exception as e:
        print("Distribution analysis failed for", name, e)

pd.DataFrame(dist_records).to_csv(os.path.join(RESULT_DIR, "prediction_distribution_summary.csv"), index=False)
print("Saved prediction distribution summary")


Saved prediction distribution summary


In [27]:
# -------------------------
# === 7) Future forecasting helper ===
# -------------------------
def forecast_future(models_to_use=None, future_features_csv=None, n_steps=14):
    """
    Forecast future n_steps.
    - future_features_csv: optional CSV path with features for future rows (must match feature_schema columns except Date)
    - If not provided, use autoregressive approach: take last row(s) and shift lags if present to create next rows (basic)
    - Returns: dict of model_name -> predictions array
    """
    results = {}
    # load custom future features if provided
    if future_features_csv is not None and os.path.exists(future_features_csv):
        future_df = pd.read_csv(future_features_csv)
        # ensure columns alignment
        future_df = future_df[FEATURES].copy()
        # transform if preprocessor exists
        if preprocessor is not None:
            X_future_pp = preprocessor.transform(future_df)
        else:
            X_future_pp = future_df.values
        for name, model in (models.items() if models_to_use is None else {k:models[k] for k in models_to_use}.items()):
            try:
                try:
                    preds = model.predict(future_df)
                except Exception:
                    preds = model.predict(X_future_pp)
                results[name] = preds
            except Exception as e:
                print("Forecast failed for", name, e)
        return results

    # else: simple autoregressive forecast using last row repeated (not sophisticated)
    last_row = X_all.iloc[-1:].copy()
    future_rows = []
    for i in range(n_steps):
        # naive: repeat last row (user should provide better future features for accurate forecasting)
        future_rows.append(last_row.iloc[0].to_dict())
    future_df = pd.DataFrame(future_rows)
    if preprocessor is not None:
        X_future_pp = preprocessor.transform(future_df)
    else:
        X_future_pp = future_df.values
    for name, model in (models.items() if models_to_use is None else {k:models[k] for k in models_to_use}.items()):
        try:
            try:
                preds = model.predict(future_df)
            except Exception:
                preds = model.predict(X_future_pp)
            results[name] = preds
        except Exception as e:
            print("Forecast failed for", name, e)
    return results

# Save a small README describing produced files
with open(os.path.join(RESULT_DIR, "README.txt"), "w") as f:
    f.write("Analysis artifacts generated on: " + datetime.utcnow().isoformat() + "\n\n")
    f.write("Files:\n")
    f.write(" - holdout_performance.csv : model metrics on holdout\n")
    f.write(" - perm_importance/*.csv : permutation importance per model\n")
    f.write(" - ablation_results.csv : ablation test results dropping feature groups\n")
    f.write(" - shap/* : shap values and images (if shap installed)\n")
    f.write(" - topk/*.csv : top-k error analysis per model\n")
    f.write(" - plots/*.html : interactive plotly plots for Actual vs Predicted and distributions\n")
    f.write("\nNotes:\n - SHAP is preferred for global+local model explainability; waterfall is a local decomposition (single sample).\n - LIME can be used for local explanations but SHAP gives consistent global attribution and faster tree explainer for tree models.\n")

print("\nAnalysis complete. Results saved under:", RESULT_DIR)


Analysis complete. Results saved under: artifacts/analysis/


In [29]:
from pathlib import Path

OUT_ROOT = Path(f"artifacts/analysis{folder_tag}")

In [30]:
SUMMARY_DIR = OUT_ROOT / "summary"; SUMMARY_DIR.mkdir(exist_ok=True)

In [32]:
PLOTS = OUT_ROOT / "plots"; PLOTS.mkdir(exist_ok=True)

In [33]:
# ---------------- Overshoot / Undershoot visualization ----------------
print("Creating overshoot/undershoot visuals")
overshoot_summary = []
for name, yhat in predictions.items():
    overs = (yhat > y_test.values).sum()
    unders = (yhat < y_test.values).sum()
    overshoot_summary.append({"Model": name, "overshoot_count": int(overs), "undershoot_count": int(unders),
                              "overshoot_prop": float(overs/len(yhat)), "undershoot_prop": float(unders/len(yhat))})
overs_df = pd.DataFrame(overshoot_summary).sort_values("overshoot_count", ascending=False)
overs_df.to_csv(SUMMARY_DIR / "overshoot_summary.csv", index=False)

# scatter Actual vs Predicted colored by overshoot/undershoot for each model (PNG + HTML)
for name, yhat in predictions.items():
    df_sc = pd.DataFrame({"Actual": y_test.values, "Predicted": yhat})
    df_sc["Type"] = np.where(df_sc["Predicted"] > df_sc["Actual"], "Overshoot", "Undershoot")
    plt.figure(figsize=(6,6))
    colors = {"Overshoot":"#d62728", "Undershoot":"#1f77b4"}
    for t, grp in df_sc.groupby("Type"):
        plt.scatter(grp["Actual"], grp["Predicted"], label=t, alpha=0.6, s=30, c=colors[t])
    mn = min(df_sc["Actual"].min(), df_sc["Predicted"].min()); mx = max(df_sc["Actual"].max(), df_sc["Predicted"].max())
    plt.plot([mn,mx],[mn,mx], linestyle="--", color="gray", linewidth=1)
    plt.xlabel("Actual"); plt.ylabel("Predicted")
    plt.title(f"{name}: Actual vs Predicted (Overshoot=red, Undershoot=blue)")
    plt.legend()
    pngp = PLOTS / f"{name}_overshoot_scatter.png"
    plt.savefig(pngp, bbox_inches="tight", dpi=200)
    plt.close()
    # plotly interactive
    if PLOTLY_AVAILABLE:
        fig = px.scatter(df_sc, x="Actual", y="Predicted", color="Type", title=f"{name}: Actual vs Predicted (Overshoot/Undershoot)",
                         hover_data=["Actual","Predicted"])
        fig.add_shape(dict(type="line", x0=mn, x1=mx, y0=mn, y1=mx, line=dict(dash="dash")))
        htmlp = PLOTS / f"{name}_overshoot_scatter.html"
        try:
            fig.write_html(htmlp)
        except Exception:
            pass
print("Saved overshoot visuals")

# Quick stacked bar for overshoot vs undershoot counts
plt.figure(figsize=(8,4))
plt.bar(overs_df["Model"], overs_df["overshoot_count"], label="Overshoot")
plt.bar(overs_df["Model"], overs_df["undershoot_count"], bottom=overs_df["overshoot_count"], label="Undershoot")
plt.xticks(rotation=45)
plt.title("Overshoot vs Undershoot counts (stacked)")
plt.ylabel("count")
plt.legend()
plt.tight_layout()
plt.savefig(PLOTS / "overshoot_vs_undershoot_stacked.png", dpi=200)
plt.close()


Creating overshoot/undershoot visuals
Saved overshoot visuals
